# NARS rule writer · LoRA on Llama-3.1-8B-Instruct

Teaches Llama-3.1-8B the four jobs DeepSeek-V3 does in the NARS pipeline: **triples**, **aliases**, **plan** and **rules**.

1. On your Mac: `python make_sft.py` → `data/sft/train.jsonl` and `data/sft/val.jsonl`.
2. Upload both files to Google Drive in `MyDrive/nars_lora/`.
3. Runtime → Change runtime type → **A100** (L4 works, about 3× slower).
4. Run with `PILOT = True` first (a few minutes): it checks everything and estimates the full run time.
5. Set `PILOT = False` and run all. Checkpoints go to Drive, so a disconnect resumes where it stopped.

In [ ]:
%%capture
!pip install unsloth

## Settings

In [ ]:
PILOT = True                     # True: 30 steps on 200 examples, then a time estimate for the full run

BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct"
DRIVE_DIR = "/content/drive/MyDrive/nars_lora"
RUN_NAME = "llama31_8b_nars_r16"  # change it for a new experiment, so runs don't overwrite each other

MAX_LEN = 4096                   # longer examples are dropped (counted below)
EPOCHS = 2
LR = 2e-4
BATCH = 2                        # per step; lower it to 1 if you run out of memory
ACCUM = 8                        # effective batch = BATCH × ACCUM = 16
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
SEED = 0

# Full run: any error in any cell writes it to Drive and disconnects the GPU, so a failed run at night
# does not keep using compute units. Stopping a cell yourself (the stop button) does not disconnect.
import traceback
def stop_on_error(shell, etype, value, tb, tb_offset=None):
    shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    if PILOT:
        return
    from google.colab import drive, runtime
    try:
        with open(f"{DRIVE_DIR}/{RUN_NAME}_error.txt", "w") as f:
            f.write("".join(traceback.format_exception(etype, value, tb)))
        drive.flush_and_unmount()
    finally:
        runtime.unassign()
get_ipython().set_custom_exc((Exception,), stop_on_error)


## Data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, collections
OUT_DIR = f"{DRIVE_DIR}/{RUN_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)

def read(name):
    with open(f"{DRIVE_DIR}/{name}.jsonl") as f:
        return [json.loads(line) for line in f]

train_rows, val_rows = read("train"), read("val")
for name, rows in [("train", train_rows), ("val", val_rows)]:
    print(name, len(rows), "examples", dict(collections.Counter(r["task"] for r in rows)))

## Model

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)

## Chat format

The Llama-3.1 template of the base model, the same one vLLM applies when it serves the adapter, so training and testing see identical prompts.

In [ ]:
from datasets import Dataset

def to_dataset(rows):
    texts = [tokenizer.apply_chat_template(r["messages"], tokenize=False) for r in rows]
    lengths = [len(tokenizer(t, add_special_tokens=False).input_ids) for t in texts]
    keep = [i for i, n in enumerate(lengths) if n <= MAX_LEN]
    print(f"{len(keep)}/{len(rows)} examples fit in {MAX_LEN} tokens · {sum(lengths[i] for i in keep):,} tokens")
    return Dataset.from_dict({"text": [texts[i] for i in keep], "task": [rows[i]["task"] for i in keep]}), \
        sum(lengths[i] for i in keep)

import random
if PILOT:
    train_rows = random.Random(SEED).sample(train_rows, 200)
by_job = collections.defaultdict(list)
for r in val_rows:
    by_job[r["task"]].append(r)
per_job = 15 if PILOT else 75                     # validation loss on a sample (a full pass costs minutes), equal per job
val_rows = [r for rows in by_job.values() for r in random.Random(SEED).sample(rows, min(per_job, len(rows)))]
train_data, train_tokens = to_dataset(train_rows)
val_data, _ = to_dataset(val_rows)
print(train_data[0]["text"][:600])

## Train

Loss counts only DeepSeek's replies, not the prompts.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_data, eval_dataset=val_data,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_LEN, packing=False, group_by_length=True,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
        num_train_epochs=EPOCHS, max_steps=30 if PILOT else -1,
        learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.03, weight_decay=0.01, optim="adamw_8bit",
        bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
        logging_steps=5 if PILOT else 10, eval_strategy="steps", eval_steps=15 if PILOT else 100,
        save_strategy="steps", save_steps=15 if PILOT else 100, save_total_limit=3,
        load_best_model_at_end=True, metric_for_best_model="eval_loss",   # keep and save the lowest validation loss
        output_dir=f"{OUT_DIR}/checkpoints", report_to="none", seed=SEED))
trainer = train_on_responses_only(trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n")

In [ ]:
import glob
resume = bool(glob.glob(f"{OUT_DIR}/checkpoints/checkpoint-*")) and not PILOT
print("resuming from the last checkpoint" if resume else "starting fresh")
stats = trainer.train(resume_from_checkpoint=resume or None)

seconds = stats.metrics["train_runtime"]
if PILOT:
    seen = 30 * BATCH * ACCUM                                   # examples the pilot trained on
    rate = train_tokens / len(train_data) * seen / seconds      # tokens per second
    full_tokens = train_tokens / len(train_data) * len(read("train")) * EPOCHS
    print(f"\n{rate:,.0f} tokens/s → the full run ({EPOCHS} epochs) takes about {full_tokens / rate / 3600:.1f} h on this GPU")
else:
    print(f"\ntrained in {seconds / 3600:.1f} h")

## Save the adapter

In [ ]:
model.save_pretrained(f"{OUT_DIR}/adapter")
tokenizer.save_pretrained(f"{OUT_DIR}/adapter")
print("adapter saved to", f"{OUT_DIR}/adapter")

## Check on validation

Greedy replies on 10 held-out examples per job, compared with DeepSeek's reply:
- **valid**: the reply parses (JSON for triples, aliases and rules)
- **triples**: overlap with DeepSeek's triples (F1); **aliases / rules**: the same groups or rules, in any order; **plan**: the same hops, ignoring spacing

A different reply can still be right, so this is a sanity check. The real test is NARS accuracy on dev.
The report is also saved to Drive as `check.md`, so you can read it after the session ends.

In [ ]:
import re

def parse(task, text):
    """The plan is plain text (spacing ignored); the other jobs must reply with a JSON list or object."""
    if task == "plan":
        return " ".join(text.split()) or None
    try:
        value = json.loads(re.sub(r"^```(json)?|```$", "", text.strip()).strip())
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, (list, dict)) else None

def score(task, got, want):
    """1.0 = same as DeepSeek; for triples, the overlap (F1)."""
    try:
        if task == "triples":
            g = {(t["subject"], t["relation"], t["object"]) for t in got}
            w = {(t["subject"], t["relation"], t["object"]) for t in want}
            return 2 * len(g & w) / (len(g) + len(w)) if g or w else 1.0
        if task == "aliases":
            return float(sorted(map(sorted, got)) == sorted(map(sorted, want)))
        if task == "rules":
            key = lambda reply: sorted(json.dumps(rule, sort_keys=True) for rule in reply["rules"])
            return float(key(got) == key(want))
        return float(got == want)
    except (TypeError, KeyError, AttributeError):
        return 0.0

FastLanguageModel.for_inference(model)
model.generation_config.max_length = None      # silences the max_length warning
by_task = collections.defaultdict(list)
for r in val_rows:
    by_task[r["task"]].append(r)

report, summary = [], []
for task in ["plan", "aliases", "triples", "rules"]:
    rows, valid, total = by_task[task][:10], 0, 0.0
    for k, r in enumerate(rows):
        prompt = tokenizer.apply_chat_template(r["messages"][:1], add_generation_prompt=True, return_tensors="pt").to("cuda")
        output = model.generate(input_ids=prompt, max_new_tokens=1024, do_sample=False)
        reply = tokenizer.decode(output[0, prompt.shape[1]:], skip_special_tokens=True)
        teacher = r["messages"][1]["content"]
        got = parse(task, reply)
        valid += got is not None
        total += score(task, got, parse(task, teacher)) if got is not None else 0.0
        if k == 0:                              # one example per job, side by side
            report += [f"\n### {task}  ({r['dataset']} {r['id']})", "**Prompt (end):**", "```", r["messages"][0]["content"][-600:], "```",
                       "**Student:**", "```", reply.strip(), "```", "**DeepSeek:**", "```", teacher.strip(), "```"]
    line = f"{task:<8} valid {valid}/{len(rows)} · same as DeepSeek {total / len(rows):.0%}"
    print(line)
    summary.append("- " + line)

with open(f"{OUT_DIR}/check.md", "w") as f:
    f.write("\n".join([f"# Check · {RUN_NAME}\n", *summary, *report]))
print("\n".join(report))
print("\nsaved to", f"{OUT_DIR}/check.md")

## End the session

Writes everything to Drive, then disconnects the GPU so it stops using compute units. Your notebook and Drive files stay.

In [ ]:
from google.colab import runtime
drive.flush_and_unmount()
runtime.unassign()